In [1]:
import torch

def run_cam_identity_experiment():
    # 1. 실험 재현성을 위한 랜덤 시드 고정
    torch.manual_seed(42)

    # 2. 가상 텐서 세팅: Batch=1, Channel=4, H=3, W=3
    B, C, H, W = 1, 4, 3, 3
    feature_map = torch.rand(B, C, H, W) * 10

    # 특정 클래스 c에 매핑된 최종 레이어의 채널별 가중치 w_k
    w_k = torch.tensor([1.5, -0.8, 2.3, 0.5])

    print("=" * 60)
    print(" [실험 5] 시그마 교환 법칙을 통한 CAM 메커니즘 등가성 증명 ")
    print("=" * 60)

    # ----------------------------------------------------------------
    # 경로 A: 표준 순전파 연산 (Feature Map -> GAP -> 가중치 선형 결합 = S_c)
    # ----------------------------------------------------------------
    # GAP: H, W 차원 평균 수행 -> (B, C)
    gap = feature_map.mean(dim=(2, 3))
    # Batch 0번째 샘플과 가중치 벡터 내적
    score_path_A = torch.dot(gap[0], w_k).item()

    # ----------------------------------------------------------------
    # 경로 B: CAM 원리 연산 (Feature Map 선형 결합 -> CAM Map 생성 -> 공간 평균)
    # ----------------------------------------------------------------
    # 기존 for 루프를 브로드캐스팅 연산으로 최적화: (C, 1, 1) * (B, C, H, W) -> (B, C, H, W)
    # 채널 차원(dim=1)을 따라 합산(sum)하여 최종 CAM Map (B, H, W) 생성
    cam_maps = (w_k.view(1, C, 1, 1) * feature_map).sum(dim=1)
    cam_map_sample = cam_maps[0] # Batch 0번째의 CAM 공간 맵

    # 생성된 CAM 맵 전체의 공간 평균 수행
    score_path_B = cam_map_sample.mean().item()

    # ----------------------------------------------------------------
    # 결과 검증 및 출력
    # ----------------------------------------------------------------
    absolute_error = abs(score_path_A - score_path_B)
    is_identical = torch.isclose(torch.tensor(score_path_A), torch.tensor(score_path_B), atol=1e-6)

    print(f"[*] 경로 A (GAP 수행 후 선형 결합) 최종 Score : {score_path_A:.6f}")
    print(f"[*] 경로 B (선형 결합 후 공간 평균) 최종 Score : {score_path_B:.6f}")
    print(f"[*] 두 수식의 절대 오차 (Absolute Error)      : {absolute_error:.8e}")
    print(f"[*] 두 연산 결과의 수학적 동등성 여부         : {is_identical.item()}")
    print("-" * 60)
    print("-> 결론: GAP는 정보의 손실을 유발하는 것이 아니라, 선형 대수의")
    print("         교환 법칙(Commutative Property)에 의해 공간 활성 정보(CAM)를")
    print("         보존한 채 최종 스코어와 완벽한 수학적 등가성을 유지함이 증명됨.")
    print("=" * 60)

if __name__ == "__main__":
    run_cam_identity_experiment()

 [실험 5] 시그마 교환 법칙을 통한 CAM 메커니즘 등가성 증명 
[*] 경로 A (GAP 수행 후 선형 결합) 최종 Score : 16.413742
[*] 경로 B (선형 결합 후 공간 평균) 최종 Score : 16.413742
[*] 두 수식의 절대 오차 (Absolute Error)      : 0.00000000e+00
[*] 두 연산 결과의 수학적 동등성 여부         : True
------------------------------------------------------------
-> 결론: GAP는 정보의 손실을 유발하는 것이 아니라, 선형 대수의
         교환 법칙(Commutative Property)에 의해 공간 활성 정보(CAM)를
         보존한 채 최종 스코어와 완벽한 수학적 등가성을 유지함이 증명됨.
